# Project 3 — CMIP6 exploration
**Two Futures, One Planet: How Our Emissions Choices Shape 2100**

Question: *How does the projected temperature gap between a low-emissions (SSP1-2.6) and a high-emissions (SSP5-8.5) future change over the century?*

Data access follows the DSC 106 course notebook (Pangeo CMIP6 catalog on Google Cloud, anonymous `gcsfs`, `xr.open_zarr`).

## Cell 1 — setup

In [ ]:
!pip install --upgrade -q xarray zarr gcsfs cftime nc-time-axis
import numpy as np, pandas as pd, xarray as xr, gcsfs
from concurrent.futures import ThreadPoolExecutor
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
gcs = gcsfs.GCSFileSystem(token="anon")
print("xarray", xr.__version__)

## Cell 2 — find (model, member, grid) triples with historical + SSP1-2.6 + SSP5-8.5 monthly `tas`

In [ ]:
cat = pd.read_csv("https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv")
EXPS = ["historical", "ssp126", "ssp585"]
KEYS = ["source_id", "member_id", "grid_label"]

tas = cat[(cat.table_id == "Amon") & (cat.variable_id == "tas") & cat.experiment_id.isin(EXPS)]
tas = (tas.sort_values("version")
          .drop_duplicates(KEYS + ["experiment_id"], keep="last"))   # newest version only

cover = tas.groupby(KEYS).experiment_id.nunique()
full = cover[cover == 3].reset_index()[KEYS]

full["rank"] = list(zip(full.member_id != "r1i1p1f1", full.grid_label != "gn", full.member_id))
short = full.sort_values("rank").drop_duplicates("source_id").drop(columns="rank")

lf_cat = cat[(cat.table_id == "fx") & (cat.variable_id == "sftlf")]
lf = lf_cat[["source_id", "grid_label"]].drop_duplicates().assign(has_sftlf=True)
short = short.merge(lf, on=["source_id", "grid_label"], how="left")
short["has_sftlf"] = short.has_sftlf.eq(True)

cands = tas.merge(short[KEYS], on=KEYS)   # catalog rows (3 per model) for Cell 3

print("catalog rows:", len(cat))
print("tas Amon rows per experiment:\n", tas.experiment_id.value_counts().to_string())
print("matched (model, member, grid) triples:", len(full), "| distinct models:", short.source_id.nunique())
print(short.sort_values("source_id").to_string(index=False))

## Cell 3 — metadata check: units, calendar, time coverage, grid (1-D vs 2-D coordinates), California-box cell count

In [ ]:
CA = dict(lat_min=32.5, lat_max=42.0, lon_min=-124.5, lon_max=-114.0)  # bounding box around California

def coord_names(ds):
    latn = next(n for n in ["lat", "latitude", "nav_lat"] if n in ds.variables)
    lonn = next(n for n in ["lon", "longitude", "nav_lon"] if n in ds.variables)
    return latn, lonn

def inspect(row):
    out = dict(source_id=row.source_id, experiment_id=row.experiment_id)
    try:
        ds = xr.open_zarr(gcs.get_mapper(row.zstore), consolidated=True)
        latn, lonn = coord_names(ds)
        lat, lon = ds[latn].values, ds[lonn].values
        out["coord_dims"] = f"{lat.ndim}D"            # check BEFORE counting cells
        if lat.ndim == 1:
            LAT, LON = np.meshgrid(lat, lon, indexing="ij")
            dlat, dlon = np.abs(np.diff(lat)).mean(), np.abs(np.diff(lon)).mean()
        else:
            LAT, LON = lat, lon
            dlat, dlon = np.abs(np.diff(lat, axis=0)).mean(), np.abs(np.diff(lon, axis=1)).mean()
        L = LON % 360
        inbox = ((LAT >= CA["lat_min"]) & (LAT <= CA["lat_max"]) &
                 (L >= CA["lon_min"] % 360) & (L <= CA["lon_max"] % 360))
        out.update(units=ds["tas"].attrs.get("units"),
                   tas_dims=",".join(ds["tas"].dims),
                   calendar=ds.time.encoding.get("calendar", "?"),
                   span=f"{str(ds.time.values[0])[:7]}→{str(ds.time.values[-1])[:7]}",
                   ntime=ds.sizes["time"],
                   nlat=LAT.shape[0], nlon=LAT.shape[1],
                   dlat=round(float(dlat), 2), dlon=round(float(dlon), 2),
                   lon_range=f"{np.nanmin(lon):.1f}..{np.nanmax(lon):.1f}", ca_cells=int(inbox.sum()))
    except Exception as e:
        out["error"] = repr(e)[:100]
    return out

with ThreadPoolExecutor(8) as ex:
    res = pd.DataFrame(list(ex.map(inspect, cands.itertuples())))
if "error" not in res: res["error"] = np.nan

spans = res.pivot(index="source_id", columns="experiment_id", values="span").add_suffix("_span")
units_all = res.groupby("source_id").units.agg(lambda s: ",".join(sorted(set(map(str, s)))))
grid = (res[res.experiment_id == "ssp585"]
          .set_index("source_id")[["coord_dims", "tas_dims", "calendar", "nlat", "nlon", "dlat", "dlon", "lon_range", "ca_cells"]])
summary = (short.set_index("source_id")
             .join(grid).join(spans).join(units_all.rename("units"))
             .sort_values("dlat"))
errs = res.dropna(subset=["error"])
if len(errs): print("ERRORS:\n", errs[["source_id", "experiment_id", "error"]].to_string())
print(summary.to_string())
summary.to_csv("stage1_model_check.csv")

## Cell 4 — choose model and scope from the Cell 3 table

Rules (applied in code, then checked by eye):
1. Complete coverage: historical 1850-01 → 2014-12, both SSPs 2015-01 → ≥ 2100-12; units `K` in all three runs; no errors.
2. California scope only if: 1-D lat/lon coordinates, grid spacing ≤ 1.3° in both directions, ≥ 40 cells in the box, and a land-fraction file (`sftlf`) exists so ocean cells can be excluded. Otherwise fall back to a global area-weighted mean.
3. Among eligible models prefer member `r1i1p1f1`, native grid `gn`, then spacing closest to 1° (enough California cells while keeping the download manageable).

In [ ]:
MODEL_OVERRIDE = None     # set to a source_id string to force a choice

s = summary.reset_index().fillna({"historical_span": "", "ssp126_span": "", "ssp585_span": ""})
first = lambda c: s[c].str[:7]; last = lambda c: s[c].str[-7:]
s["complete"] = (first("historical_span").le("1850-12") & last("historical_span").ge("2014-12") &
                 first("ssp126_span").eq("2015-01") & last("ssp126_span").ge("2100-12") &
                 first("ssp585_span").eq("2015-01") & last("ssp585_span").ge("2100-12") &
                 s.units.eq("K") & ~s.source_id.isin(errs.source_id))
s["ca_ok"] = (s.complete & s.has_sftlf & s.coord_dims.eq("1D") &
              s.dlat.le(1.3) & s.dlon.le(1.3) & s.ca_cells.ge(40))
s["rank"] = list(zip(s.member_id != "r1i1p1f1", s.grid_label != "gn", (s.dlat - 1.0).abs(), s.source_id))
print("complete models:", int(s.complete.sum()), "| California-eligible:", int(s.ca_ok.sum()))
print(s[s.ca_ok].sort_values("rank")[["source_id","member_id","grid_label","dlat","dlon","ca_cells","calendar"]].head(10).to_string(index=False))

pool = s[s.ca_ok] if s.ca_ok.any() else s[s.complete & s.coord_dims.eq("1D")]
choice = (s[s.source_id == MODEL_OVERRIDE] if MODEL_OVERRIDE else pool.sort_values("rank")).iloc[0]
MODEL, MEMBER, GRID = choice.source_id, choice.member_id, choice.grid_label
SCOPE = "california_box" if bool(choice.ca_ok) else "global"
print(f"\nCHOSEN: {MODEL} {MEMBER} {GRID} | scope={SCOPE} | dlat={choice.dlat} dlon={choice.dlon} | box cells={choice.ca_cells} | calendar={choice.calendar}")
STORES = cands[(cands.source_id == MODEL)].set_index("experiment_id").zstore.to_dict()
for k, v in STORES.items(): print(k, v)

## Cell 5 — extract regional and global series

* Spatial mean: area-weighted with cos(latitude). California: cells whose centres fall in the box **and** whose land fraction ≥ 50 %.
* Global area-weighted mean is computed in the same pass (same chunks) for context.
* Monthly → calendar-year mean weighted by days in month (model calendar). K → °C.
* Baseline: same model's historical 1995–2014 mean (IPCC AR6 reference period).

In [ ]:
def open_tas(exp):
    return xr.open_zarr(gcs.get_mapper(STORES[exp]), consolidated=True)["tas"]

ds_h = open_tas("historical").sel(time=slice("1950-01", "2014-12"))
ds_1 = open_tas("ssp126").sel(time=slice("2015-01", "2100-12"))
ds_5 = open_tas("ssp585").sel(time=slice("2015-01", "2100-12"))
for n, d in [("hist", ds_h), ("ssp126", ds_1), ("ssp585", ds_5)]:
    print(n, d.dims, d.shape, d.attrs.get("units"))
assert np.allclose(ds_1.lat, ds_5.lat) and np.allclose(ds_1.lon, ds_5.lon) and np.allclose(ds_h.lat, ds_5.lat)

lat, lon = ds_5.lat, ds_5.lon
coslat = np.cos(np.deg2rad(lat))
w_glob = coslat * xr.ones_like(lon)

if SCOPE == "california_box":
    lf_row = lf_cat[(lf_cat.source_id == MODEL) & (lf_cat.grid_label == GRID)].sort_values("version").iloc[-1]
    sftlf = xr.open_zarr(gcs.get_mapper(lf_row.zstore), consolidated=True)["sftlf"].squeeze(drop=True).load()
    frac = sftlf.values / (100.0 if float(sftlf.max()) > 1.5 else 1.0)
    land = xr.DataArray(frac >= 0.5, dims=("lat", "lon"), coords={"lat": lat, "lon": lon})
    lon360 = lon % 360
    box = ((lat >= CA["lat_min"]) & (lat <= CA["lat_max"]) &
           (lon360 >= CA["lon_min"] % 360) & (lon360 <= CA["lon_max"] % 360))
    w_reg = (coslat * box * land).astype(float)
    cells = w_reg.where(w_reg > 0, drop=True)
    print("sftlf store:", lf_row.zstore)
    print("land cells used:", int((w_reg > 0).sum()), "of", int(box.sum()), "box cells")
    print("cell-centre lats:", np.round(cells.lat.values, 2)); print("cell-centre lons:", np.round(((cells.lon + 180) % 360) - 180, 2).values)
else:
    w_reg = w_glob

def spatial(da):
    return xr.Dataset({"reg": da.weighted(w_reg).mean(("lat", "lon")),
                       "glob": da.weighted(w_glob).mean(("lat", "lon"))})

def annual(ds):
    w = ds.time.dt.days_in_month
    out = (ds * w).groupby("time.year").sum() / w.groupby("time.year").sum()
    out["n_months"] = ds.reg.groupby("time.year").count()
    return out

import time; t0 = time.time()
ann = {k: annual(spatial(d)).compute() for k, d in [("hist", ds_h), ("ssp126", ds_1), ("ssp585", ds_5)]}
print(f"computed in {time.time()-t0:.0f}s")

## Cell 6 — verify and export `data/tas_scenarios_annual.csv` (2015–2100) and `data/tas_historical_annual.csv` (1950–2014)

In [ ]:
import os, json
K = 273.15
a1, a5, ah = ann["ssp126"], ann["ssp585"], ann["hist"]
base_reg = float(ah.reg.sel(year=slice(1995, 2014)).mean()) - K
base_glob = float(ah.glob.sel(year=slice(1995, 2014)).mean()) - K

df = pd.DataFrame({
    "year": a1.year.values.astype(int),
    "ssp126_c": a1.reg.values - K,
    "ssp585_c": a5.reg.sel(year=a1.year).values - K,
})
df["gap_c"] = df.ssp585_c - df.ssp126_c
df["ssp126_anom_c"] = df.ssp126_c - base_reg
df["ssp585_anom_c"] = df.ssp585_c - base_reg
df["global_ssp126_c"] = a1.glob.values - K
df["global_ssp585_c"] = a5.glob.sel(year=a1.year).values - K
df["global_gap_c"] = df.global_ssp585_c - df.global_ssp126_c
df = df.round(3)

hist = pd.DataFrame({"year": ah.year.values.astype(int), "hist_c": ah.reg.values - K,
                     "hist_anom_c": ah.reg.values - K - base_reg, "global_hist_c": ah.glob.values - K}).round(3)

# ---- verification ----
yrs = list(range(2015, 2101))
assert df.year.tolist() == yrs, "years not exactly 2015..2100"
assert df.year.is_unique and not df.isna().any().any(), "duplicates or NaN"
assert (a1.n_months.values == 12).all() and (a5.sel(year=a1.year).n_months.values == 12).all(), "incomplete years"
assert hist.year.tolist() == list(range(1950, 2015)) and not hist.isna().any().any()
print("VERIFIED: 86 rows, years 2015-2100, 12 months each, no missing/duplicate years, no NaN")

os.makedirs("data", exist_ok=True)
df.to_csv("data/tas_scenarios_annual.csv", index=False)
hist.to_csv("data/tas_historical_annual.csv", index=False)
meta = dict(model=MODEL, member=MEMBER, grid=GRID, scope=SCOPE, region_box=CA,
            land_threshold="sftlf >= 50%" if SCOPE == "california_box" else None,
            variable="tas (near-surface air temperature), monthly Amon, converted K -> degC",
            annual="calendar-year mean weighted by days in month",
            spatial="cos(latitude) area weights", baseline="historical 1995-2014 mean",
            baseline_reg_c=round(base_reg, 3), baseline_glob_c=round(base_glob, 3),
            stores=STORES)
json.dump(meta, open("data/metadata.json", "w"), indent=2)
print(json.dumps(meta, indent=2))
print("----- tas_scenarios_annual.csv -----"); print(df.to_csv(index=False))
print("----- tas_historical_annual.csv -----"); print(hist.to_csv(index=False))

## Results of the checkpoint run (Colab, 2026-11-02)

* Catalog: 523,774 stores; `tas`/`Amon` stores: 635 historical, 329 SSP1-2.6, 327 SSP5-8.5.
* 311 (model, member, grid) triples have all three experiments, spanning 44 models; **37 models** pass the coverage/units checks; all candidate grids use **1-D** lat/lon coordinates.
* 14 models qualify for a California average (≤ 1.3° grid, ≥ 40 box cells, land-fraction file).
* **Chosen:** `CESM2-WACCM` · `r1i1p1f1` · `gn` (0.94° × 1.25°, `noleap` calendar, units K in all three runs).
  * historical `gs://cmip6/CMIP6/CMIP/NCAR/CESM2-WACCM/historical/r1i1p1f1/Amon/tas/gn/v20190227/`
  * ssp126 `gs://cmip6/CMIP6/ScenarioMIP/NCAR/CESM2-WACCM/ssp126/r1i1p1f1/Amon/tas/gn/v20210211/`
  * ssp585 `gs://cmip6/CMIP6/ScenarioMIP/NCAR/CESM2-WACCM/ssp585/r1i1p1f1/Amon/tas/gn/v20200702/`
  * land fraction (fixed field, same grid) `gs://cmip6/CMIP6/ScenarioMIP/NCAR/CESM2-WACCM/ssp534-over/r5i1p1f1/fx/sftlf/gn/v20200305/`
* **Region:** California bounding box 32.5–42°N, 124.5–114°W, land cells with land fraction ≥ 50 % → 64 of 88 cells (box also includes most of Nevada and small parts of OR/AZ/Baja). Area weights cos(lat).
* **Verification:** both scenarios have exactly one value per year 2015–2100 (86 rows), 12 months each, no NaN/duplicates. Historical 1950–2014 also complete.
* **Baseline** (1995–2014, historical run): California box 14.744 °C; global 14.740 °C.
* Exports: `data/tas_scenarios_annual.csv`, `data/tas_historical_annual.csv`, `data/metadata.json`.
